In [1]:
import scanpy as sc

adata = sc.read_h5ad('/data/scfm-mid-output/raw/XAtlasOrion_HEK293T_subset.h5ad')
adata.var_names

OSError: Unable to synchronously open file (bad object header version number)

In [1]:
import scanpy as sc
from pathlib import Path
import argparse
import gc

def main():
    
    input_dir = Path("/home/oem/scfm-layer-analysis-refactored/data/embeddings/XAtlasOrion_HCT116_subset_tahoe_1b")
    output = Path("/data/scfm-mid-output/embeddings/XAtlasOrion_HCT116_merged_tahoe_1b.h5ad")
    compression = "gzip"  # Opzioni: 'gzip', 'lzf', None

    h5ad_files = list(input_dir.glob("*.h5ad"))

    if not h5ad_files:
        raise RuntimeError(f"Nessun file .h5ad trovato in {input_dir}")

    print(f"Trovati {len(h5ad_files)} file .h5ad")
    print("Merge in corso...")

    adatas = []

    for path in h5ad_files:
        print(f"  → Carico {path.name}")
        adata = sc.read_h5ad(path)
        adatas.append(adata)

    print("Concatenazione...")
    adata_merged = sc.concat(
        adatas,
        axis=0,              # concat su obs
        join="inner",        # var identiche
        merge="same",        # uns coerente
        label="chunk_id",    # opzionale
        index_unique=None
    )

    print(f"Scrittura output -> {output}")
    output.parent.mkdir(parents=True, exist_ok=True)
    adata_merged.write_h5ad(output, compression=compression)

    # Cleanup
    del adatas, adata_merged
    gc.collect()

    print("MERGE COMPLETATO ✅")

if __name__ == "__main__":
    main()


Trovati 2 file .h5ad
Merge in corso...
  → Carico XAtlasOrion_HCT116_subset_chunk_0000.h5ad
  → Carico XAtlasOrion_HCT116_subset_chunk_0001.h5ad
Concatenazione...
Scrittura output -> /data/scfm-mid-output/embeddings/XAtlasOrion_HCT116_merged_tahoe_1b.h5ad
MERGE COMPLETATO ✅


In [2]:
import scanpy as sc

adata = sc.read_h5ad(
    "/data/scfm-mid-output/embeddings/XAtlasOrion_HCT116_merged_tahoe_1b.h5ad"
)


In [4]:
print(adata.obs['gene_target'].value_counts())

gene_target
FAM3A      30
TWNK       28
VAX1       27
CDK17      27
E2F4       27
           ..
ADAMTS8     1
ZBTB45      1
SPC24       1
GSTA3       1
ZNF596      1
Name: count, Length: 1997, dtype: int64
